In [1]:
from google import genai
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv()
client =genai.Client()

interaction = client.interactions.create(
    model= "gemini-3.7-flash",
    input="請逐步教學含視窗化操作指令, 如何使用raspberry樹莓派架設出家用的NAS伺服器?")
display(Markdown(interaction.output_text))
      

在樹莓派（Raspberry Pi）上架設 NAS，最推薦且功能最強大的「視窗化（Web GUI）解決方案」是使用 **OpenMediaVault (OMV)**。

OMV 是一套專門為 NAS 設計的開源系統，安裝完成後，你**不需要再碰任何指令**，直接用電腦或手機的瀏覽器打開「網頁控制台」，就能用滑鼠完成所有硬碟管理、權限設定與檔案分享。

以下為您整理詳細的逐步圖形化架設教學：

---

### 一、 事前硬體與軟體準備

1. **硬體需求：**
   * Raspberry Pi 4 或 Pi 5（建議 4GB/8GB 記憶體，傳輸效能較好）。
   * 16GB 以上 MicroSD 卡（當作系統碟）。
   * 外接 USB 3.0 硬碟或 SSD（建議使用有「獨立供電」的外接硬碟）。
   * 網路線（強烈建議接實體網路線以維持傳輸穩定）。
2. **軟體工具：**
   * 電腦下載並安裝：[Raspberry Pi Imager](https://www.raspberrypi.com/software/)。

---

### 二、 步驟 1：燒錄樹莓派基礎系統

OMV 是運行在無桌面版的 Debian 系統上，我們需要先燒錄乾淨的系統：

1. 打開 **Raspberry Pi Imager**。
2. **選擇裝置**：選擇你的樹莓派型號（如 Raspberry Pi 4）。
3. **選擇作業系統**：點擊 `Raspberry Pi OS (other)` $\to$ 選擇 **`Raspberry Pi OS Lite (64-bit)`**（無桌面輕量版）。
4. **選擇儲存卡**：選擇你的 MicroSD 卡。
5. 點擊「下一步」，跳出「套用 OS 自訂設定」時點選 **「編輯設定」**：
   * 設定**主機名稱**（如 `raspi-nas`）。
   * 設定**使用者名稱與密碼**（請務必記住，如 `pi` / `你的密碼`）。
   * 切換到「服務」分頁，勾選 **「啟用 SSH」**（選擇「使用密碼驗證」）。
6. 儲存設定並開始**「寫入」**SD 卡。

---

### 三、 步驟 2：一鍵安裝 OpenMediaVault (OMV)

這是整個過程**唯一**需要輸入指令的步驟，用來啟動視窗化網頁後台：

1. 將 SD 卡插入樹莓派，接上網路線並開機（等待約 2 分鐘）。
2. 在同一區域網路的電腦上打開終端機（Windows 打開 PowerShell，Mac 打開 Terminal）。
3. 輸入連線指令並按下 Enter（輸入密碼時不會顯示是正常的）：
   ```bash
   ssh pi@raspi-nas.local
   # 或者 ssh pi@<你的樹莓派IP>
   ```
4. 連線成功後，**複製並貼上以下官方一鍵安裝指令**，按下 Enter：
   ```bash
   wget -O - https://github.com/OpenMediaVault-Plugin-Developers/installScript/raw/master/install | sudo bash
   ```
5. **等待安裝**：這需要大約 15~25 分鐘（取決於網路速度），安裝完成後樹莓派會**自動重新開機**。
6. *（終端機可以關閉了，接下來全程使用視窗化操作！）*

---

### 四、 步驟 3：進入 OMV 視窗化控制台

1. 在電腦瀏覽器輸入樹莓派的 IP 位址（例如：`http://192.168.1.100` 或 `http://raspi-nas.local`）。
2. 看到 OpenMediaVault 登入畫面：
   * **預設帳號**：`admin`
   * **預設密碼**：`openmediavault`
3. 登入後，強烈建議先修改預設密碼：
   * 點選右上角 **使用者圖示** $\to$ **更改密碼**。

---

### 五、 步驟 4：硬碟掛載與格式化（視窗操作）

將外接硬碟插上樹莓派的 **USB 3.0（藍色孔）**。

1. **掛載硬碟**：
   * 點擊左側選單：**儲存區 (Storage)** $\to$ **檔案系統 (File Systems)**。
   * 點擊上方 **「+」號** $\to$ 選擇 **掛載 (Mount)**。
   * 在「檔案系統」下拉選單中，選擇你的外接硬碟 $\to$ 點擊 **儲存 (Save)**。
   * *(注意：如果硬碟是全新的，請先到「磁碟 (Disks)」進行抹除，再到「檔案系統」建立為 `EXT4` 或 `Btrfs` 格式)*。
2. **套用變更**：
   * OMV 每次修改設定，頂部會跳出黃色提示條，請務必點擊 **「黃色勾勾 (套用)」** $\to$ 確認套用。

---

### 六、 步驟 5：建立共享資料夾與使用者（視窗操作）

#### 1. 建立共享資料夾
1. 點選左側：**儲存區 (Storage)** $\to$ **共享資料夾 (Shared Folders)**。
2. 點擊 **「+ 建立 (Create)」**：
   * **名稱**：輸入你想顯示的資料夾名（例如：`Data_Share` 或 `Movies`）。
   * **檔案系統**：選擇剛才掛載的外接硬碟。
   * **相對路徑**：預設即可。
   * **權限**：選擇 `Administrator: read/write, Users: read/write, Others: no access`。
3. 點擊 **儲存 (Save)** 並套用變更（黃色勾勾）。

#### 2. 建立存取帳號
1. 點選左側：**使用者 (Users)** $\to$ **使用者 (Users)**。
2. 點擊 **「+ 建立 (Create)」**：
   * **名稱**：例如 `nasuser`。
   * **密碼**：設定一組連線密碼。
3. 點擊 **儲存 (Save)**。
4. 點擊剛才建立的使用者，點擊上方的 **「權限 (Privileges)」** 圖示：
   * 在剛建立的 `Data_Share` 資料夾旁邊，勾選 **可讀寫 (Read/Write)**。
   * 點擊 **儲存** 並套用變更。

---

### 七、 步驟 6：開啟 Windows / Mac 檔案共享 (SMB/CIFS)

1. 點選左側：**服務 (Services)** $\to$ **SMB/CIFS** $\to$ **設定 (Settings)**。
2. 將 **「已啟用 (Enabled)」** 開關打開 $\to$ 點擊 **儲存**。
3. 切換到上方分頁 **「共用 (Shares)」**：
   * 點擊 **「+ 建立 (Create)」**。
   * **已啟用**：開啟。
   * **共享資料夾**：選擇步驟 5 建立的 `Data_Share`。
   * **公開**：選擇 `不允許 (No)`（要求輸入密碼較安全）。
4. 點擊 **儲存**，並點擊頂部 **「黃色勾勾」** 套用設定。

---

### 八、 步驟 7：從電腦連線至你的 NAS

#### 【Windows 電腦連線】
1. 打開「檔案總管」，在上方網址列輸入：
   ```text
   \\你的樹莓派IP  (例如 \\192.168.1.100)
   ```
2. 輸入步驟 6 建立的使用者名稱（`nasuser`）和密碼。
3. 看到共享資料夾後，可點右鍵選擇 **「連線網路磁碟機」**，將它固定在你的「我的電腦」中（就像一個本地槽位，如 Z: 槽）。

#### 【Mac 電腦連線】
1. 打開 **Finder**，按鍵盤快捷鍵 `Command + K`（或上方選單「前往」$\to$「連接伺服器」）。
2. 輸入伺服器位址：
   ```text
   smb://你的樹莓派IP  (例如 smb://192.168.1.100)
   ```
3. 點擊「連線」，輸入使用者名稱與密碼即可掛載。

---

### 💡 進階維護小技巧（重要）
1. **設定固定 IP**：建議至家用 Wi-Fi 路由器的後台，將樹莓派的 MAC 位址綁定「固定 IP」，避免路由器重開機後 IP 跑掉導致連不上。
2. **安全關機**：若要拔除電源，請先進入 OMV 網頁後台，點擊右上角 **電源圖示 $\to$ 關機**，以防外接硬碟資料損毀。
3. **擴充功能**：日後可以在 OMV 的「系統 $\to$ 外掛程式」中安裝 Docker / Portainer，輕鬆視窗化一鍵架設 **Plex/Jellyfin (家庭影音中心)** 或 **qBittorrent (離線下載機)**。